# 20 Cell stack + stacked submerged ORR (US Claim 1)

Series current conservation; depth-varying pocket thickness equalizes air ΔP.

## Governing equations

See the requirement UID and patent claims cited in the title cell. Default electrolyte $6\,\mathrm{M}$ KOH at $303\,\mathrm{K}$ and $1\,\mathrm{atm}$. Currents in $\mathrm{A}$, voltages in $\mathrm{V}$, power in $\mathrm{W}$. Chemistry remains $\mathrm{...}$ (not mhchem).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation setup

Integrate the model once. Plot sections below each document what is shown, why it matters for patent/requirement verification, and the equations that govern that view.


In [ ]:
from plant_sim.substructures.cell_stack import CellStack

st = CellStack(P, n_cells=3)
eq = st.equal_pressure_drop(compensate=True)
nq = st.equal_pressure_drop(compensate=False)

## Cell-stack equal-ΔP ORR pockets

### What this plot illustrates

Stack model (decade 20) verifies US Claim 1: submerged ORR pockets share approximately equal air-side pressure drop.

### Governing equations

$\Delta P_i(Q_i,L_i,A_{h,i})$ equalized by depth-varying geometry so $Q_i$ remains balanced down the stack.


In [ ]:
plot_frame = pd.DataFrame(
    {
        "x": (eq["z_m"]),
        "dP_comp": (eq["dP_Pa"]),
        "dP_naive": (nq["dP_Pa"]),
    }
)
plot_long = plot_frame.melt(id_vars="x", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="x", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("depth (m)")
ax.set_ylabel("ΔP (Pa)")
ax.set_title("Equal ΔP verification")
ax.text(0.0, -0.22, "US12308414B2 Claim 1", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

## Numerical checks

Finite-trajectory and capacity checks for the integration above.


In [ ]:
res = st.simulate((0, 180), n_eval=40)
print("stack finite", np.all(np.isfinite(res["y"])), "n_cells", res["n_cells"])